# A Reinforcement Learning (RL) environment and agent for a simplified architectural space utilization task.

We will use Q-learning (tabular RL) to train an agent to place three distinct programmatic zones—Living, Circulation (Hallway), and Service (Bathroom)—into a $1 \times 3$ grid.

The Problem

DesignGoal: Place all three zones without overlapping.

Bonus: Reward the agent for placing the Circulation zone in the middle slot (optimal architectural flow).

Penalty: Punish the agent for overlapping zones or leaving the grid incomplete.

In [ ]:
import numpy as np
import random

class SpacePlanningEnv:
    def __init__(self):
        # 3 grid slots (0, 1, 2).
        # State represents what is in each slot: 0=Empty, 1=Living, 2=Circulation, 3=Service
        self.state = (0, 0, 0)
        self.zones = [1, 2, 3] # Living, Circulation, Service

    def reset(self):
        self.state = (0, 0, 0)
        return self.state

    def step(self, action):
        """
        Action is a tuple: (zone_id, slot_index)
        e.g., (1, 0) means place Living (1) in Slot 0.
        """
        zone, slot = action
        current_layout = list(self.state)

        # Scenario 1: Overwrite an existing placement (Invalid design step)
        if current_layout[slot] != 0:
            reward = -10
            done = True
            return self.state, reward, done

        # Place the zone
        current_layout[slot] = zone
        self.state = tuple(current_layout)

        # Check if all zones are placed
        placed_zones = [z for z in self.state if z != 0]

        if len(placed_zones) == 3:
            done = True
            # Base reward for a completed, non-overlapping layout
            reward = 10

            # Architectural adjacency preference: Circulation (2) in the middle slot (1)
            if self.state[1] == 2:
                reward += 15  # Optimization bonus
        else:
            done = False
            reward = -1  # Step penalty to encourage speed

        return self.state, reward, done

# --- Agent Training Layout ---

# Initialize environment and Q-table
env = SpacePlanningEnv()

# Action space: 3 zones * 3 slots = 9 possible actions
# Mapping index to actual action tuple
action_space = [(zone, slot) for zone in env.zones for slot in range(3)]
num_actions = len(action_space)

# Q-Table dictionary: keys are states (tuples), values are np.zeros(num_actions)
q_table = {}

def get_q_values(state):
    if state not in q_table:
        q_table[state] = np.zeros(num_actions)
    return q_table[state]

# Hyperparameters
alpha = 0.1    # Learning rate
gamma = 0.9    # Discount factor
epsilon = 0.3  # Exploration rate
episodes = 2000

# Training Loop
for episode in range(episodes):
    state = env.reset()
    done = False

    while not done:
        # Epsilon-greedy action selection
        if random.uniform(0, 1) < epsilon:
            action_idx = random.randint(0, num_actions - 1)
        else:
            action_idx = np.argmax(get_q_values(state))

        action = action_space[action_idx]

        # Step environment
        next_state, reward, done = env.step(action)

        # Update Q-value
        old_value = get_q_values(state)[action_idx]
        next_max = np.max(get_q_values(next_state))

        new_value = (1 - alpha) * old_value + alpha * (reward + gamma * next_max)
        q_table[state][action_idx] = new_value

        state = next_state

# --- Evaluation / Results ---
print("--- Training Complete ---")
print(f"Total explored states: {len(q_table)}\n")

print("Testing the trained agent layout strategy:")
state = env.reset()
done = False
steps = []

# Mapping numbers to human-readable names
zone_mapping = {0: "Empty", 1: "Living", 2: "Circulation", 3: "Service"}

while not done:
    action_idx = np.argmax(get_q_values(state))
    action = action_space[action_idx]
    steps.append(f"Placed {zone_mapping[action[0]]} in slot {action[1]}")
    state, reward, done = env.step(action)

print(f"Final Grid State: {[zone_mapping[z] for z in state]}")
print("Agent actions:")
for step in steps:
    print(f"  -> {step}")

--- Training Complete ---
Total explored states: 64

Testing the trained agent layout strategy:
Final Grid State: ['Living', 'Circulation', 'Living']
Agent actions:
  -> Placed Circulation in slot 1
  -> Placed Living in slot 0
  -> Placed Living in slot 2


##How to Scale This

Because layout design involves complex spatial constraints, tabular Q-learning hits limits quickly.

To scale this for real architectural workflows:

- Continuous Coordinates: Replace the simple grid with a 2D bounding box coordinate system ($x, y, w, h$) and use a Deep Q-Network (DQN) or Proximal Policy Optimization (PPO).

- Complex Rewards: Add penalties for blocking window vectors (daylighting), or rewards for compact plumbing walls (Service zone adjacency).